# Week 68, Day 1: The C API: an overview

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w068d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import math, json, sys

for f in (math.sqrt, json.dumps, len, print):
    kind = type(f).__name__
    print(f"{f.__name__:6} {kind}")

print(type(math).__name__, getattr(math, "__file__", "built into the interpreter")[-12:])

In [ ]:
import re

c_source = '''
static PyMethodDef fastmath_methods[] = {
    {"hypot", fastmath_hypot, METH_VARARGS, "Length of the hypotenuse."},
    {"clamp", fastmath_clamp, METH_VARARGS | METH_KEYWORDS, "Limit a value."},
    {NULL, NULL, 0, NULL}
};
'''
entries = re.findall(r'\{"(\w+)",\s*(\w+),\s*([\w| ]+),', c_source)
for name, c_function, flags in entries:
    print(f"Python name {name!r} -> C function {c_function} ({flags.strip()})")

In [ ]:
def parse(fmt, args):
    "A simplified PyArg_ParseTuple: returns the converted values or raises."
    required, _, optional = fmt.partition("|")
    codes = required + optional
    if not len(required) <= len(args) <= len(codes):
        raise TypeError(f"function takes {len(required)} to {len(codes)} arguments ({len(args)} given)")
    out = []
    for code, value in zip(codes, args):
        if code == "d":
            if not isinstance(value, (int, float)):
                raise TypeError(f"must be real number, not {type(value).__name__}")
            out.append(float(value))
        elif code == "i":
            if isinstance(value, float) or not isinstance(value, int):
                raise TypeError(f"'{type(value).__name__}' object cannot be interpreted as an integer")
            out.append(value)
        elif code == "s":
            if not isinstance(value, str):
                raise TypeError(f"argument must be str, not {type(value).__name__}")
            out.append(value)
    return out

print(parse("dd", (3, 4.5)), parse("s|i", ("tea",)))
for bad in [("dd", (3,)), ("i", (2.5,)), ("s", (42,))]:
    try:
        parse(*bad)
    except TypeError as e:
        print("TypeError:", e)

In [ ]:
counts = {}
owned = []

def new_ref(name):                    # e.g. PyLong_FromLong: we own it
    counts[name] = counts.get(name, 0) + 1
    owned.append(name)

def decref(name):
    counts[name] -= 1
    owned.remove(name)
    if counts[name] == 0:
        print("  freed", name)

new_ref("total")
new_ref("label")
decref("label")                        # finished with it
print("still owned:", owned, "(we must DECREF or return 'total')")

In [ ]:
class CError:
    "The current exception, as CPython keeps it per thread."
    current = None

def c_divide(a, b):
    if b == 0:
        CError.current = ZeroDivisionError("division by zero")
        return None                         # NULL
    return a / b

def c_average(values):
    total = 0.0
    for v in values:
        part = c_divide(v, len(values))
        if part is None:                     # check every result...
            return None                      # ...and pass the error up
        total += part
    return total

def call_from_python(func, *args):
    result = func(*args)
    if result is None and CError.current is not None:
        error, CError.current = CError.current, None
        raise error
    return result

print(call_from_python(c_average, [2, 4, 6]))
def c_ratio(a, b):
    part = c_divide(a, b)
    return None if part is None else round(part, 3)

try:
    call_from_python(c_ratio, 1, 0)
except ZeroDivisionError as e:
    print("ZeroDivisionError reached Python:", e)

> **Key idea.** An extension module is C code that includes Python.h, defines functions taking and returning PyObject *, lists them in a PyMethodDef table, describes the module in a PyModuleDef, and exposes PyInit_name, which import calls. PyArg_ParseTuple converts the argument tuple using a format string (i, l, d, s, p, O, | for optional), setting TypeError or OverflowError on a mismatch. Reference counts are managed by hand: new references are owned and must be released (Py_DECREF) or passed on; borrowed references mustn't be released and must be Py_INCREFed to keep; some functions steal references. Errors set an exception and return NULL or -1, and every caller must check and pass it up. Release the GIL around long C work with Py_BEGIN_ALLOW_THREADS. Build with setuptools; the limited API gives abi3 wheels for many Python versions.

### Exercise: Parsing arguments like C does

Write `parse_args(fmt, args)`, a simplified `PyArg_ParseTuple` returning a list of converted values. Codes: `i` (an `int`, not a `bool` or `float`, from -2147483648 to 2147483647, otherwise `OverflowError("signed integer is out of range")`), `d` (an `int` or `float`, returned as `float`; `bool` counts as an `int`), `s` (a `str`), `p` (anything, converted with `bool`), and `O` (anything, unchanged). A `|` marks where optional arguments begin. Wrong counts raise `TypeError("expected N to M arguments, got K")` (or `"expected N arguments, got K"` when there's nothing optional); wrong types raise `TypeError` naming the code, like `"argument 2 must be d, not str"`.

In [ ]:
def parse_args(fmt, args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_parse():\n    \"PyArg_ParseTuple\"\n    assert parse_args(\"dd\", (3, 4.5)) == [3.0, 4.5] and parse_args(\"s|ip\", (\"tea\",)) == [\"tea\"]\n    assert parse_args(\"s|ip\", (\"tea\", 2, [])) == [\"tea\", 2, False] and parse_args(\"O\", ([1],)) == [[1]]\n    assert parse_args(\"d\", (True,)) == [1.0]\n    cases = [\n        (\"dd\", (3,), TypeError, \"expected 2 arguments, got 1\"),\n        (\"s|i\", (), TypeError, \"expected 1 to 2 arguments, got 0\"),\n        (\"di\", (1.0, 2.5), TypeError, \"argument 2 must be i, not float\"),\n        (\"i\", (True,), TypeError, \"argument 1 must be i, not bool\"),\n        (\"i\", (2 ** 31,), OverflowError, \"signed integer is out of range\"),\n        (\"s\", (b\"x\",), TypeError, \"argument 1 must be s, not bytes\"),\n    ]\n    for fmt, args, err, message in cases:\n        try:\n            parse_args(fmt, args)\n            assert False, fmt\n        except err as e:\n            assert str(e) == message, str(e)")

### Exercise: Who owns this reference?

Audit a sequence of simulated C API calls for reference counting mistakes. Each operation is a tuple: `("new", name)` (a function returned a new reference: we own one), `("borrow", name)` (we hold a borrowed reference: we own nothing), `("incref", name)`, `("decref", name)`, `("steal", name)` (we passed our reference to a function that steals it), and `("return", name)` (we return it to the caller, passing ownership). Write `audit(ops)`: a dict with `"leaks"` (sorted names we still own references to at the end, each once) and `"errors"` (a list, in order, of messages: `"decref of borrowed x"` when we `decref` or `steal` or `return` something we don't own).

In [ ]:
def audit(ops):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_refs():\n    \"new, borrowed and stolen references\"\n    good = [(\"new\", \"total\"), (\"new\", \"label\"), (\"decref\", \"label\"), (\"return\", \"total\")]\n    assert audit(good) == {\"leaks\": [], \"errors\": []}\n    leaky = [(\"new\", \"a\"), (\"new\", \"b\"), (\"new\", \"a\"), (\"decref\", \"a\")]\n    assert audit(leaky) == {\"leaks\": [\"a\", \"b\"], \"errors\": []}\n    wrong = [(\"borrow\", \"item\"), (\"decref\", \"item\"), (\"new\", \"x\"), (\"steal\", \"x\"), (\"decref\", \"x\")]\n    assert audit(wrong) == {\"leaks\": [], \"errors\": [\"decref of borrowed item\", \"decref of borrowed x\"]}\n    kept = [(\"borrow\", \"item\"), (\"incref\", \"item\"), (\"return\", \"item\")]\n    assert audit(kept) == {\"leaks\": [], \"errors\": []}\n    assert audit([(\"borrow\", \"y\"), (\"return\", \"y\")]) == {\"leaks\": [], \"errors\": [\"decref of borrowed y\"]}")

### Exercise: Does it fit in C?

C integers have fixed sizes (Week 65's NumPy pitfall, again). Using the table `LIMITS` of C type names, write `limits(ctype)`: the `(minimum, maximum)` of a signed type of `bits` bits (`-(2 ** (bits - 1))` to `2 ** (bits - 1) - 1`) or an unsigned one (`0` to `2 ** bits - 1`); `fits(value, ctype)`: True if a Python int fits; and `smallest_type(values)`: the first type in `ORDER` that every value fits in, or `None`.

In [ ]:
LIMITS = {"int8": (8, True), "uint8": (8, False), "int16": (16, True), "uint16": (16, False),
          "int32": (32, True), "uint32": (32, False), "int64": (64, True), "uint64": (64, False)}
ORDER = ["uint8", "int8", "uint16", "int16", "uint32", "int32", "uint64", "int64"]

def limits(ctype):
    ___

def fits(value, ctype):
    ___

def smallest_type(values):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fit():\n    \"C integer ranges\"\n    assert limits(\"int8\") == (-128, 127) and limits(\"uint16\") == (0, 65535) and limits(\"int64\")[1] == 2 ** 63 - 1\n    assert fits(255, \"uint8\") and not fits(256, \"uint8\") and not fits(-1, \"uint32\") and fits(-2 ** 31, \"int32\")\n    assert smallest_type([0, 200]) == \"uint8\" and smallest_type([-5, 100]) == \"int8\" and smallest_type([-1, 200]) == \"int16\"\n    assert smallest_type([2 ** 70]) is None and smallest_type([]) == \"uint8\"")

### Exercise: The error convention

Simulate the C API's way of reporting errors. `ErrorState` holds the current exception: `set(exc)`, `occurred()` (the exception or `None`) and `clear()`. Write `c_parse_int(text, state)`: return the integer, or set `ValueError(f"invalid literal: {text!r}")` and return `None` (C's `NULL`); `c_sum_parsed(texts, state)`: parse each with `c_parse_int`, **checking every result** and returning `None` at once (leaving the exception set) if one fails; and `to_python(func, *args)`: call a C-style function with a fresh `ErrorState` and either return its result or raise the exception it set. A function that returns `None` without setting an exception is itself a bug: raise `SystemError("error return without exception set")`, as CPython does.

In [ ]:
class ErrorState:
    def __init__(self):
        ___

def c_parse_int(text, state):
    ___

def c_sum_parsed(texts, state):
    ___

def to_python(func, *args):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_errors():\n    \"NULL plus a set exception\"\n    s = ErrorState()\n    assert c_parse_int(\"12\", s) == 12 and s.occurred() is None\n    assert c_parse_int(\"x\", s) is None and isinstance(s.occurred(), ValueError)\n    s.clear()\n    assert s.occurred() is None\n    assert to_python(c_sum_parsed, [\"1\", \"2\", \"3\"]) == 6\n    try:\n        to_python(c_sum_parsed, [\"1\", \"oops\", \"3\"])\n        assert False\n    except ValueError as e:\n        assert str(e) == \"invalid literal: 'oops'\"\n    def careless(state):\n        return None\n    try:\n        to_python(careless)\n        assert False\n    except SystemError as e:\n        assert str(e) == \"error return without exception set\"")

### Exercise: Reading a method table

Write `method_table(c_source)`: find the `PyMethodDef` entries in C source (lines like `{"hypot", fastmath_hypot, METH_VARARGS, "Doc."},`) with a regular expression, and return a list of dicts with `name`, `function`, `flags` (a list of the `METH_` flags, in order) and `doc` (the docstring, or `None` if it's `NULL`), stopping at the `{NULL, ...}` end marker. Then `module_name(c_source)`: the name from `PyInit_name`, or `None`.

In [ ]:
import re

def method_table(c_source):
    ___

def module_name(c_source):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_methods():\n    \"the parts of an extension module\"\n    src = \"\"\"\nstatic PyMethodDef geo_methods[] = {\n    {\"hypot\", geo_hypot, METH_VARARGS, \"Length of the hypotenuse.\"},\n    {\"clamp\", geo_clamp, METH_VARARGS | METH_KEYWORDS, NULL},\n    {\"origin\", geo_origin, METH_NOARGS, \"The point (0, 0).\"},\n    {NULL, NULL, 0, NULL}\n};\nPyMODINIT_FUNC\nPyInit_geo(void)\n{\n    return PyModule_Create(&geo_module);\n}\n\"\"\"\n    table = method_table(src)\n    assert [m[\"name\"] for m in table] == [\"hypot\", \"clamp\", \"origin\"]\n    assert table[1] == {\"name\": \"clamp\", \"function\": \"geo_clamp\", \"flags\": [\"METH_VARARGS\", \"METH_KEYWORDS\"], \"doc\": None}\n    assert table[2][\"doc\"] == \"The point (0, 0).\" and module_name(src) == \"geo\"\n    assert method_table(\"int x;\") == [] and module_name(\"int x;\") is None")

### Bug hunt: A leak in a loop

Each pass through the loop gets a **new** reference to a temporary object and never releases it, so the audit finds a leak: this prints `['tmp']`. Release the temporary at the end of each pass so it prints `[]`.

In [ ]:
owned = {}

def new(name):
    owned[name] = owned.get(name, 0) + 1

def decref(name):
    owned[name] -= 1

for i in range(3):
    new("tmp")
    # ... use tmp ...

print(sorted(n for n, c in owned.items() if c > 0))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"release what you own\"\n    expect_output(\"[]\")")

### Bug hunt: A float where C wanted an int

The format string says `i` for an argument that's meant to be a price in pounds, which can have pence, so passing `2.5` fails with `TypeError`. The C function really wants a `double`: use `d` so this prints `[2.5, 3]`.

In [ ]:
def parse(fmt, args):
    out = []
    for code, value in zip(fmt, args):
        if code == "i" and not isinstance(value, int):
            raise TypeError(f"'{type(value).__name__}' object cannot be interpreted as an integer")
        out.append(float(value) if code == "d" else value)
    return out

print(parse("ii", (2.5, 3)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the right format code\"\n    expect_output(\"[2.5, 3]\")")

### Bug hunt: An error ignored

`c_total` calls a C-style helper that returns `None` (NULL) on failure, but doesn't check the result, so it carries on with `None` and crashes later with a confusing `TypeError` instead of the real problem. Check the result and return `None` at once, so the caller sees the real error and this prints `ValueError: bad value: 'x'`.

In [ ]:
error = []

def c_value(text):
    if not text.isdigit():
        error.append(ValueError(f"bad value: {text!r}"))
        return None
    return int(text)

def c_total(texts):
    total = 0
    for t in texts:
        total += c_value(t)
    return total

result = c_total(["1", "x"])
if result is None:
    print(f"{type(error[0]).__name__}: {error[0]}")

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"check every result\"\n    expect_output(\"ValueError: bad value: 'x'\")")

### Puzzle: A C function's body

Arrange the lines of the body of a C extension function that adds two doubles (as you'd write it in C). One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # the format code for a C double
q = ___   # what a C API function returning PyObject * returns on failure, as a string
r = ___   # "new" or "borrowed": the reference PyList_GetItem returns
s = ___   # True or False: you must Py_DECREF a borrowed reference when you're done with it
t = ___   # the C function import calls to create a module named fastmath

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == \"d\" and q == \"NULL\" and r == \"borrowed\", \"p, q and r\"\n    assert s is False and t == \"PyInit_fastmath\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a pretend extension module

Put today's rules together in a simulation of what an extension module gives Python. Write `extension(name, doc, methods)`, which returns a real module object (`types.ModuleType`) whose functions behave like C functions:

- `methods` is a list of `(python_name, format, implementation, docstring)`. `implementation(*c_values, state)` is a "C function": it receives the values `parse_args` produced and an `ErrorState`, and returns a result or `None` after setting an exception.
- Each module function parses its positional arguments with `parse_args(format, args)` (copy your version from the first exercise, keeping its errors), calls the implementation, and turns `None` plus an exception into a raised exception (or `SystemError("error return without exception set")` when no exception was set). Keyword arguments raise `TypeError(f"{python_name}() takes no keyword arguments")`, as `METH_VARARGS` functions do.
- Each function's `__name__` and `__doc__` come from the table; the module's `__doc__` is `doc`, and `module.__methods__` lists the names in order.
- `calls(module)` returns a dict from each function name to how many times it has been called successfully.

In [ ]:
import types

class ErrorState:
    def __init__(self):
        self._exc = None
    def set(self, exc):
        self._exc = exc
    def occurred(self):
        return self._exc

def parse_args(fmt, args):
    ___

def extension(name, doc, methods):
    ___

def calls(module):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_extension():\n    \"a module of C-style functions\"\n    import math\n    def c_hypot(x, y, state):\n        return math.sqrt(x * x + y * y)\n    def c_divide(a, b, state):\n        if b == 0:\n            state.set(ZeroDivisionError(\"division by zero\"))\n            return None\n        return a / b\n    def c_broken(state):\n        return None\n    geo = extension(\"geo\", \"Geometry helpers.\", [\n        (\"hypot\", \"dd\", c_hypot, \"Length of the hypotenuse.\"),\n        (\"divide\", \"dd\", c_divide, \"Divide two numbers.\"),\n        (\"broken\", \"\", c_broken, None),\n    ])\n    assert geo.__name__ == \"geo\" and geo.__doc__ == \"Geometry helpers.\" and geo.__methods__ == [\"hypot\", \"divide\", \"broken\"]\n    assert geo.hypot(3, 4) == 5.0 and geo.hypot.__name__ == \"hypot\" and geo.hypot.__doc__ == \"Length of the hypotenuse.\"\n    assert geo.divide(1, 4) == 0.25\n    for call, err, message in [(lambda: geo.divide(1, 0), ZeroDivisionError, \"division by zero\"),\n                               (lambda: geo.hypot(3), TypeError, \"expected 2 arguments, got 1\"),\n                               (lambda: geo.hypot(\"3\", 4), TypeError, \"argument 1 must be d, not str\"),\n                               (lambda: geo.hypot(x=3, y=4), TypeError, \"hypot() takes no keyword arguments\"),\n                               (lambda: geo.broken(), SystemError, \"error return without exception set\")]:\n        try:\n            call()\n            assert False\n        except err as e:\n            assert str(e) == message\n    assert calls(geo) == {\"hypot\": 1, \"divide\": 1, \"broken\": 0}")

## Warm-up

`math.sqrt`, `len` and `print` are `builtin_function_or_method`: functions written in C, not Python. `json.dumps` is a Python function (though `json` speeds itself up with a C helper module behind the scenes). The C API is what lets C code define functions and types that look exactly like Python ones.

## A whole extension module

Here is a complete extension module in C, defining one function, `fastmath.hypot(x, y)`. Read it slowly: every C extension has these parts.

```
#define PY_SSIZE_T_CLEAN
#include <Python.h>
#include <math.h>

/* 1. The function: always takes and returns PyObject pointers. */
static PyObject *
fastmath_hypot(PyObject *self, PyObject *args)
{
    double x, y;
    if (!PyArg_ParseTuple(args, "dd", &x, &y))   /* 2. parse two floats */
        return NULL;                             /*    on failure, an exception is already set */
    return PyFloat_FromDouble(sqrt(x * x + y * y));   /* 3. a NEW reference to a Python float */
}

/* 4. The method table: Python name, C function, calling convention, docstring. */
static PyMethodDef fastmath_methods[] = {
    {"hypot", fastmath_hypot, METH_VARARGS, "Length of the hypotenuse."},
    {NULL, NULL, 0, NULL}                        /* the end marker */
};

/* 5. The module definition. */
static struct PyModuleDef fastmath_module = {
    PyModuleDef_HEAD_INIT, "fastmath", "Fast maths.", -1, fastmath_methods
};

/* 6. The initialisation function, found by name when Python imports the module. */
PyMODINIT_FUNC
PyInit_fastmath(void)
{
    return PyModule_Create(&fastmath_module);
}
```

Compiled into a shared library (`fastmath.cpython-313-x86_64-linux-gnu.so` on Linux, `.pyd` on Windows), it imports like any module: `import fastmath; fastmath.hypot(3, 4)` gives `5.0`. Week 64's import system finds it with the extension-module loader and calls `PyInit_fastmath`.

## Arguments: PyArg_ParseTuple

A `METH_VARARGS` function receives its positional arguments as one Python tuple. **`PyArg_ParseTuple(args, format, &var1, &var2, ...)`** checks them against a **format string** and stores C values in the variables:

| code | Python value | C value |
|---|---|---|
| `i` | `int` (must fit a C `int`) | `int` |
| `l`, `L` | `int` | `long`, `long long` |
| `d` | `float` (or anything with `__float__`, such as an `int`) | `double` |
| `s` | `str` | `const char *` (UTF-8, read-only) |
| `p` | anything (its truth value) | `int` (0 or 1) |
| `O` | any object | `PyObject *` (a borrowed reference) |
| `\|` | the rest are optional | |

If the arguments don't match, it sets a `TypeError` (or `OverflowError` for a number too big) and returns 0, and the function returns `NULL` straight away. `PyArg_ParseTupleAndKeywords` does the same with keyword names. For speed, modern extensions often use `METH_FASTCALL` (an array of arguments, no tuple) and **Argument Clinic**, a tool CPython itself uses to generate the parsing code from a declaration.

## Reference counting rules

In C, nothing manages reference counts for you (Week 62's reference counting is done by hand, with `Py_INCREF(obj)` and `Py_DECREF(obj)`). Every C API function's documentation says what it does with references, in one of three ways:

- It returns a **new reference**: you **own** it, and you must `Py_DECREF` it when you're done (or pass ownership on, for example by returning it). `PyFloat_FromDouble`, `PyLong_FromLong`, `PyList_New`, `PyObject_GetAttrString` and most functions that create or look things up return new references.
- It returns a **borrowed reference**: you may use it, but you **don't own** it, so you mustn't `Py_DECREF` it, and it's only valid while its owner keeps it alive. `PyList_GetItem`, `PyTuple_GetItem` and `PyDict_GetItem` return borrowed references. To keep one beyond that, `Py_INCREF` it first.
- It **steals** a reference you pass in: it takes over your ownership. `PyList_SetItem` and `PyTuple_SetItem` steal; after calling them you no longer own that reference.

Getting this wrong is the classic C extension bug. A missing `Py_DECREF` is a **memory leak**; an extra one frees an object that's still in use, which crashes later in some unrelated place. Tools such as debug builds of Python, `sys.gettotalrefcount()` there, and `Py_XDECREF` (which ignores `NULL`) help; so does writing extensions in Cython or Rust (later this week), which handle reference counts for you.

## Errors, the GIL and building

**Errors** follow one convention throughout the C API: a function that fails **sets an exception** (with `PyErr_SetString(PyExc_ValueError, "...")` or by calling something that set one) and returns an **error indicator**: `NULL` for functions returning `PyObject *`, `-1` for functions returning `int`. The caller checks the result and, if it's the indicator, returns its own indicator at once, so the exception travels up to Python code, which sees an ordinary exception. Forgetting to check a result is another classic bug.

**The GIL** (Week 66): C code that doesn't touch Python objects for a while (a long computation, a blocking system call) should release it so other threads can run:

```
Py_BEGIN_ALLOW_THREADS
result = expensive_c_work(buffer, n);   /* no Python objects in here */
Py_END_ALLOW_THREADS
```

**Building**: a `setuptools` project with an `Extension("fastmath", ["fastmath.c"])` in `setup.py` (or `ext-modules` in `pyproject.toml`), built with `pip install .` or `python -m build`, which runs your platform's C compiler. The result is tied to one Python version unless you use the **limited API** (`Py_LIMITED_API`), a subset of the C API whose **stable ABI** lets one compiled file (an `abi3` wheel) work on every later Python 3 version.

## Practice

## Quiz

## Daily challenge

## Before you go

On your own computer, if you have a C compiler (on Linux and macOS you probably do; on Windows, the free "Build Tools for Visual Studio"), save the `fastmath.c` module above, add a `setup.py` with `setuptools.setup(ext_modules=[setuptools.Extension("fastmath", ["fastmath.c"])])`, and run `pip install .`. Tomorrow: calling C libraries directly from Python, with no C code of your own, using `ctypes`.